In [1]:
import cv2
import numpy as np

In [2]:
video_path = r"D:\Duy Toan\Python\DUT AI Club\Homework\day 28 3-3-2026\dataset.mp4"

In [3]:
def background_subtraction(video_path):
    cap = cv2.VideoCapture(video_path)

    backSub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        fg_mask = backSub.apply(frame)
        _, fg_mask = cv2.threshold(fg_mask, 250, 255, cv2.THRESH_BINARY)
        kernel = np.ones((5, 5), np.uint8)
        fg_mask = cv2.dilate(fg_mask, kernel, iterations=2)
        fg_mask = cv2.erode(fg_mask, kernel, iterations=3)

        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < 500:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        background_img = backSub.getBackgroundImage()
        cv2.imshow('MOG2 Detection', frame)
        cv2.imshow('MOG2 Mask', fg_mask)
        if background_img is not None:
            cv2.imshow('Estimated Background', background_img)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

background_subtraction(video_path)